# Kepler exoplanet classification - Notebook 02

**Classical baselines: Logistic Regression and XGBoost**

Author: Atilla Ahmed

---

## Abstract

This notebook establishes classical machine learning baselines for the three-class Kepler Object of Interest classification task. We train Logistic Regression and XGBoost on all three feature configurations defined in Notebook 01 (leaky, semi-leaky, leak-free), quantify the accuracy drop induced by removing label-adjacent features, and produce a reference performance table that deep learning models will need to beat in Notebook 03.

## Table of contents

1. [Setup and data loading](#1-setup-and-data-loading)
2. [Logistic Regression](#2-logistic-regression)
3. [XGBoost](#3-xgboost)
4. [Baselines comparison](#4-baselines-comparison)
5. [Summary](#5-summary)

## 1. Setup and data loading

We load the processed splits from Notebook 01, apply feature scaling with a training-only-fit `StandardScaler`, and define the evaluation protocol used consistently for all models.

### 1.1. Imports and configuration

In [1]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

from xgboost import XGBClassifier

warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)
pd.set_option("display.precision", 4)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

PROCESSED_PATH = Path("../data/processed")

int_to_class = {0: "CONFIRMED", 1: "CANDIDATE", 2: "FALSE POSITIVE"}
class_order = ["CONFIRMED", "CANDIDATE", "FALSE POSITIVE"]
colors = ["green", "orange", "red"]
print("Completed")

Completed


## 1.2. Load processed data

We read the three parquet splits produced by Notebook 01. Each split contains the features and a `target` column with integer labels (`0`=CONFIRMED, `1`=CANDIDATE, `2`=FALSE POSITIVE). We also load the three feature-set definitions from `feature_sets.json`.

In [4]:
train_df = pd.read_parquet(PROCESSED_PATH / "train.parquet")
val_df   = pd.read_parquet(PROCESSED_PATH / "val.parquet")
test_df  = pd.read_parquet(PROCESSED_PATH / "test.parquet")

with open(PROCESSED_PATH / "feature_sets.json", "r") as f:
    feature_sets = json.load(f)

def split_features_target(df):
    return df.drop(columns=["target"]), df["target"]

X_train, y_train = split_features_target(train_df)
X_val,   y_val   = split_features_target(val_df)
X_test,  y_test  = split_features_target(test_df)

print(f"Train:      X={X_train.shape}, y={y_train.shape}")
print(f"Validation: X={X_val.shape},   y={y_val.shape}")
print(f"Test:       X={X_test.shape},  y={y_test.shape}\n")

print(f"Feature sets loaded: {list(feature_sets.keys())}")
for name, features in feature_sets.items():
    valid = [c for c in features if c in X_train.columns]
    print(f"  {name}: {len(valid)} features")

Train:      X=(6694, 101), y=(6694,)
Validation: X=(1435, 101),   y=(1435,)
Test:       X=(1435, 101),  y=(1435,)

Feature sets loaded: ['setup_a_leaky', 'setup_b_semi_leaky', 'setup_c_leak_free', 'all_features']
  setup_a_leaky: 55 features
  setup_b_semi_leaky: 55 features
  setup_c_leak_free: 51 features
  all_features: 101 features


### 1.3. Feature sets

The processed data does not contain `koi_score` (dropped during cleaning in Notebook 01), so the semi-leaky configuration is now identical to the fully leaky one. We therefore compare only two configurations: `setup_leaky` (including the four false-positive flags) and `setup_leak_free` (excluding them). This 4-column difference is exactly what produced the 94% → 86% accuracy drop demonstrated in Notebook 01, Section 4.2.

In [6]:
feature_sets_effective = {
    "setup_leaky":     [c for c in feature_sets["setup_a_leaky"] if c in X_train.columns],
    "setup_leak_free": [c for c in feature_sets["setup_c_leak_free"] if c in X_train.columns],
}

for name, features in feature_sets_effective.items():
    print(f"{name}: {len(features)} features")

setup_leaky: 55 features
setup_leak_free: 51 features
